# 02464 – Human Memory Mini Project: analysis notebook

This notebook analyses the CSV files produced by `memory_experiment.py`. Keep the notebook, Python file and generated CSV files together in the repository root; no folders are required.

It covers:
- free recall
- serial recall
- memory span / capacity
- finger tapping
- articulatory suppression
- **wordsporg / idiom memory** (real idioms vs fake idioms)

The experiment runner now handles most of the boring stuff automatically:
- fixed condition order
- automatic trial generation
- automatic instruction screens
- automatic response logging
- automatic CSV export


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Flat repository: read CSV files from the same directory as this notebook.
DATA_DIR = Path.cwd()

trial_files = sorted(DATA_DIR.glob("*_trials.csv"))
item_files = sorted(DATA_DIR.glob("*_items.csv"))

print("Trial files:", len(trial_files))
print("Item files:", len(item_files))

if len(trial_files) == 0 or len(item_files) == 0:
    raise FileNotFoundError("No data files found beside the notebook. Run memory_experiment.py first and keep the generated CSV files here.")

trials = pd.concat([pd.read_csv(f) for f in trial_files], ignore_index=True)
items = pd.concat([pd.read_csv(f) for f in item_files], ignore_index=True)

print("Participants:", trials["participant"].nunique())
display(trials.head())
display(items.head())


## Bootstrap confidence intervals

The project description asks for an error measure.  
Here we use a simple bootstrap over trials.


In [ ]:
def bootstrap_mean_ci(values, n_boot=5000, ci=95, seed=42):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boots.append(sample.mean())
    alpha = (100 - ci) / 2
    return np.percentile(boots, [alpha, 100 - alpha])


# 1. Free recall


In [ ]:
free_trials = trials[trials["experiment"] == "free_recall"].copy()
free_items = items[items["experiment"] == "free_recall"].copy()

display(
    free_trials.groupby("condition")
    .agg(
        n_trials=("trial", "count"),
        mean_free_accuracy=("free_accuracy", "mean"),
        mean_primacy=("primacy_index", "mean"),
        mean_recency=("recency_index", "mean")
    )
)


In [ ]:
sp = (
    free_items.groupby(["condition", "serial_position"], as_index=False)["free_recalled"]
    .mean()
)

for condition, d in sp.groupby("condition"):
    plt.figure(figsize=(8, 4))
    plt.plot(d["serial_position"], d["free_recalled"], marker="o")
    plt.xlabel("Serial position")
    plt.ylabel("Proportion recalled")
    plt.ylim(0, 1)
    plt.title(f"Free recall serial-position curve: {condition}")
    plt.grid(alpha=0.25)
    plt.show()


In [ ]:
for effect in ["primacy_index", "recency_index"]:
    rows = []
    for condition, d in free_trials.groupby("condition"):
        lo, hi = bootstrap_mean_ci(d[effect].dropna())
        rows.append({
            "condition": condition,
            "effect": effect,
            "mean": d[effect].mean(),
            "CI_low": lo,
            "CI_high": hi
        })
    display(pd.DataFrame(rows))


# 2. Serial recall: capacity


In [ ]:
cap = trials[
    (trials["experiment"] == "serial_recall") &
    (trials["condition"].str.startswith("capacity_n"))
].copy()

cap["list_length"] = cap["condition"].str.extract(r"(\d+)").astype(int)
cap["whole_sequence_correct"] = (cap["serial_accuracy"] == 1).astype(int)

cap_summary = (
    cap.groupby("list_length", as_index=False)
    .agg(
        mean_letter_accuracy=("serial_accuracy", "mean"),
        mean_whole_sequence_accuracy=("whole_sequence_correct", "mean"),
        n_trials=("trial", "count")
    )
)
display(cap_summary)

plt.figure(figsize=(8, 4))
plt.plot(cap_summary["list_length"], cap_summary["mean_letter_accuracy"], marker="o", label="Letter-position accuracy")
plt.plot(cap_summary["list_length"], cap_summary["mean_whole_sequence_accuracy"], marker="o", label="Whole-sequence accuracy")
plt.xlabel("List length")
plt.ylabel("Proportion correct")
plt.ylim(0, 1.05)
plt.title("Working-memory capacity")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# 3. Serial recall: baseline vs tapping vs articulatory suppression


In [ ]:
serial_main = trials[
    (trials["experiment"] == "serial_recall") &
    (trials["condition"].isin(["baseline", "finger_tapping", "articulatory_suppression"]))
].copy()

serial_summary = (
    serial_main.groupby("condition", as_index=False)
    .agg(
        mean_serial_accuracy=("serial_accuracy", "mean"),
        n=("serial_accuracy", "count"),
        mean_tap_count=("tap_count", "mean")
    )
)
display(serial_summary)


# 4. Error analysis


In [ ]:
err = items[
    (items["experiment"] == "serial_recall") &
    (items["condition"].isin(["baseline", "finger_tapping", "articulatory_suppression"])) &
    (items["substitution"].notna()) &
    (items["substitution"].astype(str).str.len() > 0)
].copy()

pair_counts = (
    err.groupby(["condition", "presented_item", "substitution"])
    .size()
    .reset_index(name="count")
    .sort_values(["condition", "count"], ascending=[True, False])
)

display(pair_counts.head(40))


# 6. Wordsporg / idiom block


In [ ]:
idiom_trials = trials[trials["experiment"] == "idiom_serial_recall"].copy()

idiom_summary = (
    idiom_trials.groupby("condition", as_index=False)
    .agg(
        mean_serial_accuracy=("serial_accuracy", "mean"),
        mean_free_accuracy=("free_accuracy", "mean"),
        n_trials=("trial", "count")
    )
)
display(idiom_summary)

plt.figure(figsize=(6, 4))
plt.bar(idiom_summary["condition"], idiom_summary["mean_serial_accuracy"])
plt.ylim(0, 1)
plt.ylabel("Mean proportion correct in correct position")
plt.title("Real vs fake idioms")
plt.xticks(rotation=0)
plt.show()


In [ ]:
rows = []
for condition, d in idiom_trials.groupby("condition"):
    lo, hi = bootstrap_mean_ci(d["serial_accuracy"].dropna())
    rows.append({
        "condition": condition,
        "mean_serial_accuracy": d["serial_accuracy"].mean(),
        "CI_low": lo,
        "CI_high": hi
    })
display(pd.DataFrame(rows))


# 7. Participant-level overview


In [ ]:
participant_summary = (
    trials.groupby(["participant", "experiment", "condition"], as_index=False)
    .agg(
        n_trials=("trial", "count"),
        free_accuracy=("free_accuracy", "mean"),
        serial_accuracy=("serial_accuracy", "mean"),
        primacy=("primacy_index", "mean"),
        recency=("recency_index", "mean")
    )
)
display(participant_summary)


# 8. Notes for the report

You now have a cleaner write-up path:

## Free recall
- baseline serial-position curve
- primacy and recency indices
- compare baseline vs fast rate
- compare baseline vs unfilled delay
- compare baseline vs working-memory interference

## Serial recall
- capacity curve across list lengths
- baseline vs finger tapping
- baseline vs articulatory suppression
- substitution / error patterns

## Wordsporg / idioms
Treat this as an extra block:
- compare real idioms vs fake idioms
- report exact-position accuracy
- mention that this is an extra exploratory chunking-like manipulation if it is not strictly required by the assignment
